# Omnichannel Consumer Behavior Analysis

This historical analysis combines anonymized physical-visit and digital-activity data to examine behavior surrounding visits to mobile-carrier retail locations.

## Analytical workflow

- Standardize event timestamps across UTC offsets
- Identify target retail visits and participating panelists
- Join physical visits with app, web, and shopping activity
- Classify activities using exact and fuzzy term matching
- Compare activity occurring before, during, and after store visits

## Data availability

The original cleaned source files are not included. File paths below are retained as reproducibility placeholders and should be updated to point to appropriately authorized data.

## Limitations

This is an observational behavioral analysis. Temporal proximity does not establish that a store visit caused a digital action, or vice versa. Search-term dictionaries and fuzzy matching rules may also introduce classification error.


In [ ]:
'''
IPython.core.interactiveshell - 

    - enables full display output
    - will show each line
'''
# import libraries 
from IPython.core.interactiveshell import InteractiveShell    # interactive shell import
InteractiveShell.ast_node_interactivity = "all"    # iPython shell - display all output


import numpy as np    # numpy library
import pandas as pd    # pandas library
import datetime as dt    # date time

### Visits Data

In [ ]:
'''
Visits/Location Data

    - import visits.txt, TSV, low memory import
    - StartTime - converted to date/time object
    - UTCOffset - transformed into pos integers, conv to time delta
    - StartTime_offset - add StartTime with UTCOffset
    - 
'''


# import visits TSV file
df_visits_og = pd.read_csv('consumer_behavior_sample/visits.txt', delimiter="\t", low_memory = True)    # original visits file

df_visits = df_visits_og.copy()    # copy for manipulation



# start time
df_visits.StartTime = pd.to_datetime(df_visits.StartTime)


# UTC Offset
df_visits.UTCOffset=df_visits.UTCOffset*-1
df_visits['UTCOffset']=pd.to_timedelta(df_visits['UTCOffset'],unit='h')
df_visits['StartTime_offset']=df_visits['StartTime'] + df_visits['UTCOffset']


# unix start time variable
df_visits['Unix_StartTime'] = df_visits.StartTime_offset.apply(lambda x : (x-dt.datetime(1970,1,1)).total_seconds())


df_visits_copy = df_visits.copy()


'''
Convert LocationName 
'''
# Make all the strings lower case
df_visits.LocationName=df_visits.LocationName.str.lower() 


df_visits_verizon = df_visits[df_visits['LocationName'].fillna("").str.contains('verizon')]
df_visits_verizon['Location_clean'] = "Verizon"

df_visits_tmobile = df_visits[df_visits['LocationName'].fillna("").str.contains('tmobile|t-mobile')]
df_visits_tmobile['Location_clean'] = "T-Mobile"

df_visits_att = df_visits[df_visits['LocationName'].fillna("").str.contains('AT&T|at&t')]
df_visits_att['Location_clean'] = "AT&T"


df_visits_other = df_visits[~((df_visits['LocationName'].fillna("").str.contains("verizon")) | (df_visits['LocationName'].fillna("").str.contains("tmobile|t-mobile")) | (df_visits['LocationName'].fillna("").str.contains("AT&T|at&t")))]
df_visits_other['Location_clean'] = "other"

# Stack the 4 DataFrames
df_visits_all = df_visits_verizon.append([df_visits_tmobile, df_visits_att,df_visits_other])



In [ ]:
df_visits_all.groupby('Location_clean').PanelistId.count()
df_visits_all.groupby('Location_clean').PanelistId.nunique()

In [ ]:
#df_visits_all[df_visits_all.Categories.str.contains('Mobile',na=False)].shape
#df_visits_all[df_visits_all.Categories.str.contains('Mobile',na=False)].LocationName.unique()

'''
Get target visits (only Verizon, T-Mobile, AT&T)
'''

df_visits_all[~df_visits_all.Location_clean.str.contains('other',na=False)].shape
df_target_visits = df_visits_all[~df_visits_all.Location_clean.str.contains('other',na=False)]
df_target_pids = df_target_visits[['PanelistId']]


'''
Get Only Location Data that applies to mobile store visitor panelists
'''

df_visits_all['Visitor'] = df_visits_all['PanelistId'].isin(df_target_pids['PanelistId']) 
df_visits_all['Unix_DepartureTime'] = df_visits_all['Unix_StartTime'] + df_visits_all['VisitDuration']
df_visits_all.groupby('Visitor').PanelistId.count()


# PID check
print('\nTotal Unique PIDs -', df_visits_all.PanelistId.nunique())



In [ ]:
#clean out to just those who visited a mobile (verizion, at&t, T-mobile) store
df_visits_all = df_visits_all[(df_visits_all['Visitor']==True)]
print('visit data')
print('shape - ', df_visits_all.shape)
print('pid - ', df_visits_all.PanelistId.nunique())
df_visits_all.head()

In [ ]:
#df_visits_all[df_visits_all.Categories.str.contains('Mobile',na=False)].shape
#df_visits_all[df_visits_all.Categories.str.contains('Mobile',na=False)].LocationName.unique()

'''
Get Target visits (only Verizon, T-Mobile, AT&T)
'''

df_visits_all[~df_visits_all.Location_clean.str.contains('other',na=False)].shape
df_target_visits=df_visits_all[~df_visits_all.Location_clean.str.contains('other',na=False)]
df_target_visits

## Digital Data

In [ ]:
'''
Importing Digital Data
'''
path = './consumer_behavior_sample/'

# app data
# df_app = pd.read_csv('app.txt', delimiter="\t", low_memory = False)
df_app = pd.read_csv(path+'app.txt', delimiter="\t", low_memory = True)
print('App Data: ', df_app.shape)

# df_web = pd.read_csv('web.txt', delimiter="\t", low_memory = False)
df_web = pd.read_csv(path+'web.txt', delimiter="\t", low_memory = True)
print('Web Data: ', df_web.shape)

# shopper data
df_shopper = pd.read_csv(path+'shopper.txt', delimiter="\t", low_memory = False)
print('Shopper Data: ', df_shopper.shape)


In [ ]:
df_dig_all = pd.concat([df_app,df_web,df_shopper])
df_dig_all.PanelistId.nunique()

In [ ]:
'''
Cleaning to just those who visited one of the stores
'''
print('Joining Survey/Digital Data')
# app data
df_app['Visitor'] = df_app['PanelistId'].isin(df_target_pids['PanelistId'])

print('App Data: ', df_app.shape)
print('App Unique PID: ', df_app.PanelistId.nunique())


# shopper data
df_shopper['Visitor'] = df_shopper['PanelistId'].isin(df_target_pids['PanelistId'])

print('Shopper Data: ', df_shopper.shape)
print('Shopper Unique PID: ', df_shopper.PanelistId.nunique())


# web data
df_web['Visitor'] = df_web['PanelistId'].isin(df_target_pids['PanelistId'])

print('Web Data: ', df_web.shape)
print('Web Unique PID: ', df_web.PanelistId.nunique())


In [ ]:
#Grab only the visitors digital data
#App
df_app = df_app[(df_app['Visitor']==True)]
print('app data')
print('shape - ', df_app.shape)
print('pid - ', df_app.PanelistId.nunique())

#Web
df_web = df_web[(df_web['Visitor']==True)]
print('Web data')
print('shape - ', df_web.shape)
print('pid - ', df_web.PanelistId.nunique())

#Shopper
df_shopper = df_shopper[(df_shopper['Visitor']==True)]
print('Shop data')
print('shape - ', df_shopper.shape)
print('pid - ', df_shopper.PanelistId.nunique())


In [ ]:
df_shopper.head()

In [ ]:
df_shopper.Retailer.unique()

In [ ]:
df_target_visits_cleaned=df_target_visits[['PanelistId','VisitDuration','LocationName','StartTime_offset','Unix_StartTime','Unix_DepartureTime','Location_clean']]

df_target_visits_cleaned.head()

In [ ]:
#merge in visit data to digital to do match
'''
Merging digital with survey data
'''
print('Joining Visit/Digital Data')
# app data
df_app_join = pd.merge(
    left=df_app,
    right=df_target_visits_cleaned,
    how='left',
    left_on='PanelistId', 
    right_on='PanelistId')

df_app_join_target = df_app_join[df_app_join['PanelistId'].notnull()]

print('App Data: ', df_app_join_target.shape)
print('App Unique PID: ', df_app_join_target.PanelistId.nunique())


# web data
df_web_join = pd.merge(
    left=df_web,
    right=df_target_visits_cleaned,
    how='left',
    left_on='PanelistId', 
    right_on='PanelistId')

df_web_join_target = df_web_join[df_app_join['PanelistId'].notnull()]

print('Web Data: ', df_web_join_target.shape)
print('Web Unique PID: ', df_web_join_target.PanelistId.nunique())


# shopper data
df_shopper_join = pd.merge(
    left=df_shopper,
    right=df_target_visits_cleaned,
    how='left',
    left_on='PanelistId', 
    right_on='PanelistId')

df_shopper_join_target = df_shopper_join[df_app_join['PanelistId'].notnull()]

print('Shopper Data: ', df_shopper_join_target.shape)
print('Shopper Unique PID: ', df_shopper_join_target.PanelistId.nunique())

In [ ]:
df_web_join_target.head()

In [ ]:
'''
Convert data types
'''
print('\nConverting Date/Time Objects')
# convert Start Date
df_app_join_target.StartTime = pd.to_datetime(df_app_join_target.StartTime)
df_shopper_join_target.StartTime = pd.to_datetime(df_shopper_join_target.StartTime)
df_web_join_target.StartTime = pd.to_datetime(df_web_join_target.StartTime)


# Check the DataTypes of each columns
print('App - StartTime - ', df_app_join_target.StartTime.dtypes)
print('Shopper - StartTime - ', df_shopper_join_target.StartTime.dtypes)
print('Web - StartTime - ', df_web_join_target.StartTime.dtypes)


'''
TimeDelta for UTC Offset
'''

# app data
df_app_join_target.UTCOffset = df_app_join_target.UTCOffset*-1
df_app_join_target['UTCOffset'] = pd.to_timedelta(df_app_join_target['UTCOffset'],unit='h')
df_app_join_target['StartTimeEvent_offset'] = df_app_join_target['StartTime'] + df_app_join_target['UTCOffset']


# web data
df_web_join_target.UTCOffset = df_web_join_target.UTCOffset*-1
df_web_join_target['UTCOffset'] = pd.to_timedelta(df_web_join_target['UTCOffset'],unit='h')
df_web_join_target['StartTimeEvent_offset'] = df_web_join_target['StartTime'] + df_web_join_target['UTCOffset']


# shopper data
df_shopper_join_target.UTCOffset = df_shopper_join_target.UTCOffset*-1
df_shopper_join_target['UTCOffset'] = pd.to_timedelta(df_shopper_join_target['UTCOffset'],unit='h')
df_shopper_join_target['StartTimeEvent_offset'] = df_shopper_join_target['StartTime'] + df_shopper_join_target['UTCOffset']



'''
Unix StartTime each table
'''

df_app_join_target['Unix_StartTimeEvent_offset'] = df_app_join_target.StartTimeEvent_offset.apply(lambda x : (x-dt.datetime(1970,1,1)).total_seconds())
df_shopper_join_target['Unix_StartTimeEvent_offset'] = df_shopper_join_target.StartTimeEvent_offset.apply(lambda x : (x-dt.datetime(1970,1,1)).total_seconds())
df_web_join_target['Unix_StartTimeEvent_offset'] = df_web_join_target.StartTimeEvent_offset.apply(lambda x : (x-dt.datetime(1970,1,1)).total_seconds())


'''
Creating Digital TimeDelta
'''

# SurveyDigitalDelta
df_app_join_target['delta_digital'] = df_app_join_target['Unix_StartTime'] - df_app_join_target['Unix_StartTimeEvent_offset']
df_web_join_target['delta_digital'] = df_web_join_target['Unix_StartTime'] - df_web_join_target['Unix_StartTimeEvent_offset']
df_shopper_join_target['delta_digital'] = df_shopper_join_target['Unix_StartTime'] - df_shopper_join_target['Unix_StartTimeEvent_offset']



'''
Cleaning columns
'''
'''
# app data
col_order = ['PanelistId','StartTime_x_offset','Unix_StartTime_x_offset','SessionDuration', 'AppName','BundleId', 'Category', 'OsName', 'OsVersion', 'DeviceManufacturer','DeviceModel', 'DeviceType', 'LocationName_x', 'Location_clean','StartTime_y','Unix_StartTime', 'VisitDuration', 'Unix_DepartureTime','TCWArrived', 'TCWArrived_datetime','delta_survey-digital']
df_app= df_app[col_order]

col_name = ['PanelistId', 'StartTime_Event','Unix_StartTime_Event','SessionDuration_Event', 'AppName','BundleId', 'Category', 'OsName', 'OsVersion', 'DeviceManufacturer','DeviceModel', 'DeviceType', 'LocationName', 'Location_clean','StartTime_Visit','Unix_StartTime_Visit', 'VisitDuration', 'Unix_DepartureTime_Visit','TCWArrived', 'TCWArrived_datetime','delta_survey-digital']
df_app.columns = col_name



# web data
col_order = ['PanelistId', 'StartTime_x_offset', 'Unix_StartTime_x_offset', 'PageDuration', 'PageDomain','Category', 'SearchTerm', 'RefDomain', 'OsName', 'OsVersion','DeviceManufacturer', 'DeviceModel', 'DeviceType', 'BrowserVersion','LocationName_x', 'Location_clean', 'StartTime_y', 'Unix_StartTime', 'VisitDuration','Unix_DepartureTime',  'TCWArrived','TCWArrived_datetime', 'delta_survey-digital']
df_web = df_web[col_order]

col_name = ['PanelistId', 'StartTime_Event', 'Unix_StartTime_Event', 'PageDuration', 'PageDomain','Category', 'SearchTerm', 'RefDomain', 'OsName', 'OsVersion','DeviceManufacturer', 'DeviceModel', 'DeviceType', 'BrowserVersion','LocationName', 'Location_clean', 'StartTime_Visit', 'Unix_StartTime_Visit', 'VisitDuration','Unix_DepartureTime_Visit',  'TCWArrived','TCWArrived_datetime', 'delta_survey-digital']
df_web.columns = col_name



# shopper data
col_order = ['PanelistId', 'StartTime_x_offset','Unix_StartTime_x_offset', 'EventType', 'ProductName','ProductCode', 'ProductUrl', 'Category', 'Price', 'Quantity','Retailer', 'SearchTerm', 'OsName', 'OsVersion', 'DeviceManufacturer','DeviceModel', 'DeviceType', 'LocationName_x', 'Location_clean','StartTime_y','Unix_StartTime', 'VisitDuration', 'Unix_DepartureTime','TCWArrived', 'TCWArrived_datetime','delta_survey-digital']
df_shopper = df_shopper[col_order]

col_name = ['PanelistId', 'StartTime_Event','Unix_StartTime_Event', 'EventType', 'ProductName','ProductCode', 'ProductUrl', 'Category', 'Price', 'Quantity','Retailer', 'SearchTerm', 'OsName', 'OsVersion', 'DeviceManufacturer','DeviceModel', 'DeviceType', 'LocationName', 'Location_clean','StartTime_Visit','Unix_StartTime_Visit', 'VisitDuration', 'Unix_DepartureTime_Visit','TCWArrived', 'TCWArrived_datetime','delta_survey-digital']
df_shopper.columns = col_name
'''



'''
Add 24 hours after the departure visit
'''

df_app_join_target['Post_Unix_DepartureTime'] = df_app_join_target.Unix_DepartureTime + (3600*24)
df_web_join_target['Post_Unix_DepartureTime'] = df_web_join_target.Unix_DepartureTime + (3600*24)
df_shopper_join_target['Post_Unix_DepartureTime'] = df_shopper_join_target.Unix_DepartureTime + (3600*24)




## Activities - digital data

In [ ]:
'''
Activity Tag
'''

# app
activity_list = []
for i, r in df_app_join_target.iterrows():
    
    if (r['delta_digital'] <= (3600*24)) and (r['delta_digital'] > 0):
        activity_list.append("Pre")
    elif (r['Unix_StartTimeEvent_offset'] >= r['Unix_StartTime']) and (r['Unix_StartTimeEvent_offset'] <= r['Unix_DepartureTime'] ):  
        activity_list.append("During")
    elif (r['Unix_StartTimeEvent_offset'] > r['Unix_DepartureTime'] ) and (r['Unix_StartTimeEvent_offset'] <= r['Post_Unix_DepartureTime']): 
        activity_list.append("Post")
    else:
        activity_list.append("Other")
df_app_join_target['activity'] =  activity_list

print('app unique activities-', df_app_join_target.activity.unique())



# web
activity_list = []
for i, r in df_web_join_target.iterrows():
    
    if (r['delta_digital'] <= (3600*24)) and (r['delta_digital'] > 0):
        activity_list.append("Pre")
    elif (r['Unix_StartTimeEvent_offset'] >= r['Unix_StartTime']) and (r['Unix_StartTimeEvent_offset'] <= r['Unix_DepartureTime'] ):  
        activity_list.append("During")
    elif (r['Unix_StartTimeEvent_offset'] > r['Unix_DepartureTime'] ) and (r['Unix_StartTimeEvent_offset'] <= r['Post_Unix_DepartureTime']): 
        activity_list.append("Post")
    else:
        activity_list.append("Other")    
df_web_join_target['activity'] =  activity_list

print('web unique activities-', df_web_join_target.activity.unique())



# shopper
activity_list = []
for i, r in df_shopper_join_target.iterrows():
    
    if (r['delta_digital'] <= (3600*24)) and (r['delta_digital'] > 0):
        activity_list.append("Pre")
    elif (r['Unix_StartTimeEvent_offset'] >= r['Unix_StartTime']) and (r['Unix_StartTimeEvent_offset'] <= r['Unix_DepartureTime'] ):  
        activity_list.append("During")
    elif (r['Unix_StartTimeEvent_offset'] > r['Unix_DepartureTime'] ) and (r['Unix_StartTimeEvent_offset'] <= r['Post_Unix_DepartureTime']): 
        activity_list.append("Post")
    else:
        activity_list.append("Other")    
df_shopper_join_target['activity'] =  activity_list   

print('shopper unique activities-', df_shopper_join_target.activity.unique())



'''
reporting activities
'''
print('app - shape - ' ,df_app_join_target.shape)
print('web - shape - ' ,df_web_join_target.shape)
print('shop - shape - ' ,df_shopper_join_target.shape)

## Activities - physical visits data

In [ ]:
'''
Physical Visits data
'''
# setting order
col_order = ['PanelistId', 'StartTime','UTCOffset','StartTime_offset', 'Unix_StartTime', 'VisitDuration', 'LocationName']
df_visits_all = df_visits_all[col_order]


# column name
col_name = ['PanelistId', 'StartTime_Event','UTCOffset_Event','StartTime_offset_Event', 'Unix_StartTime_Event', 'VisitDuration_Event', 'LocationName_Event']
df_visits_all.columns = col_name


In [ ]:
# merging data
visits_join = pd.merge(
    left=df_visits_all,
    right=df_target_visits_cleaned, 
    how='left', 
    left_on= 'PanelistId', 
    right_on='PanelistId')

print('Visits Data: ', visits_join.shape)
print('Visits Unique PID: ', visits_join.PanelistId.nunique())


# Coding delta_survey event
visits_join['delta_event'] = visits_join['Unix_StartTime'] - visits_join['Unix_StartTime_Event']


'''
Cleaning merged visit data
'''

# Get only the PIDs that are in the survey
visits_join = visits_join[visits_join['PanelistId'].notnull()]

# Add Post Departure Time
visits_join['Post_Unix_DepartureTime'] = visits_join['Unix_DepartureTime']+(3600*24)

# delta_survey-event
visits_join['delta_event'] = visits_join['Unix_StartTime'] - visits_join['Unix_StartTime_Event']




In [ ]:
visits_join.columns

In [ ]:
'''
Visit activities
'''

# visits activity list
activity_list = []
for i, r in visits_join.iterrows():
    
    if (r['delta_event'] <= (3600*24)) and (r['delta_event'] > 0):
        activity_list.append("Pre")
    elif (r['Unix_StartTime_Event'] >= r['Unix_StartTime']) and (r['Unix_StartTime_Event'] <= r['Unix_DepartureTime'] ):  
        activity_list.append("During")
    elif (r['Unix_StartTime_Event'] > r['Unix_DepartureTime'] ) and (r['Unix_StartTime_Event'] <= r['Post_Unix_DepartureTime']): 
        activity_list.append("Post")
    else:
        activity_list.append("Other")    
visits_join['activity'] =  activity_list

print('visits - activities - ', visits_join.activity.unique())
print('visits - shape - ', visits_join.shape)


### Checkpoint - Count by Activity

In [ ]:
# app data check

df_app_join_target.groupby('activity').count()
df_app_join_target.groupby('activity').PanelistId.nunique()

In [ ]:
# web data check

df_web_join_target.groupby('activity').count()
df_web_join_target.groupby('activity').PanelistId.nunique()

In [ ]:
# shop data check

df_shopper_join_target.groupby('activity').count()
df_shopper_join_target.groupby('activity').PanelistId.nunique()

In [ ]:
# visits data check

visits_join.groupby('activity').count()
visits_join.groupby('activity').PanelistId.nunique()

## Filtering to Activity != "Other"

In [ ]:
'''
df_app_join_survey_panel.shape

# create checkpoint pre filter
df_app_join_survey_panel_prefilter = df_app_join_survey_panel.copy()
df_web_join_survey_panel_prefilter = df_web_join_survey_panel.copy()
df_shopper_join_survey_panel_prefilter = df_shopper_join_survey_panel.copy()
visits_survey_join_prefilter = visits_survey_join.copy()
'''

In [ ]:
#App
df_app_join_target = df_app_join_target[~(df_app_join_target['activity']=='Other')]
print('app data')
print('shape - ', df_app_join_target.shape)
print('pid - ', df_app_join_target.PanelistId.nunique())

#Web
df_web_join_target = df_web_join_target[~(df_web_join_target['activity']=='Other')]
print('Web data')
print('shape - ', df_web_join_target.shape)
print('pid - ', df_web_join_target.PanelistId.nunique())


#Shopper
df_shopper_join_target = df_shopper_join_target[~(df_shopper_join_target['activity']=='Other')]
print('Shop data')
print('shape - ', df_shopper_join_target.shape)
print('pid - ', df_shopper_join_target.PanelistId.nunique())


#Visits
visits_join = visits_join[~(visits_join['activity']=='Other')]
print('Visit data')
print('shape - ', visits_join.shape)
print('pid - ', visits_join.PanelistId.nunique())




In [ ]:
dig_all = pd.concat([df_app_join_target, df_web_join_target, df_shopper_join_target])

dig_all.shape
dig_all.PanelistId.nunique()
dig_all.groupby('activity').count()
dig_all.groupby('activity').PanelistId.nunique()


## Filter Terms


**FUZZY MATCH MODIFICATIONS**

'''
    Version updates - old code
    - add in matching specification -
            - match = 'simple'
            - match = 'partial'
            - match = 'token ratio'
    - Format - IF match = y, do a match, IF match = None
'''

    df.AppName.fillna("",inplace=True)
     simple_ratio_score = []
        partial_ratio_score = []
    token_ratio_score = []
    for i,r in df.iterrows():
        
        str1=r.AppName
        str2 = user_search_term
         Ratio = fuzz.ratio(str1.lower(),str2)
         Partial_Ratio = fuzz.partial_ratio(str1.lower(),str2)
        token_ratio = fuzz.token_set_ratio(str1.lower(),str2)
         simple_ratio_score.append(Ratio)
         partial_ratio_score.append(Partial_Ratio)
        token_ratio_score.append(token_ratio)
     df['simple_ratio_score'] = simple_ratio_score  
     df['partial_ratio_score'] = partial_ratio_score  
    df['token_ratio_score'] = token_ratio_score  



In [ ]:
# import fuzzywuzzy library
from fuzzywuzzy import fuzz

'''
Fuzzy Match function
'''
def fuzzymatch(dataframe, search_terms, source_column):
    
    # fuzzy match
    search_terms =  [x.lower() for x in applist]
    search_terms = '|'.join(search_terms)
    
    
    # fuzzy match application
    dataframe[str(source_column)].fillna("", inplace=True)
    
    token_ratio_score = []
    
    for i, r in dataframe.iterrows():
        str1= r[str(source_column)]
        str2 = search_terms
        token_ratio = fuzz.token_set_ratio(str1.lower(),str2)
        token_ratio_score.append(token_ratio)
    
    dataframe['token_ratio_score'] = token_ratio_score
    
    return dataframe



'''
Fuzzy Match filter
'''
def event_fuzz_recode(dataframe, source_column, error, new_column):
    
    if str(new_column) in dataframe.columns:
        dataframe.pop(str(new_column))
    
    
    dataframe.loc[dataframe[str(source_column)] >= int(error), str(new_column)] = 1
    dataframe[str(new_column)] = dataframe[str(new_column)].fillna(0)



## REVISED ACTIVITY

## APP - APPName

In [ ]:
df_app_join_target.head(2)
df_web_join_target.head(2)
df_shopper_join_target.head(2)
visits_join.head(2)

In [ ]:
'''
APP - AppName
'''

#create list
applist = [
    'AMZN Mobile LLC',
    'Walmart',
    'Walmart (Jet)',
    'Best Buy Co., Inc.',
    'Alibaba.com Hong Kong Limited',
    'Costco Wholesale Corporation',
    'TJ198 Network Technology(HK) Limited',
    'PCM, Inc.',
    'Newegg Inc.',
    'eBay Inc.',
    'Office Depot/OfficeMax',
    'Staples',
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
]

# lower lists
applist =  [x.lower() for x in applist]

# apply app list
mask = df_app_join_target.AppName.str.contains('|'.join(applist), na=False)
df_app_search_Appname_exact = df_app_join_target.loc[mask]

df_app_search_Appname_exact.head()
df_app_search_Appname_exact.shape




'''
APP - BundleID
'''
# apply app list
mask = df_app_join_target.BundleId.str.contains('|'.join(applist), na=False)
df_app_search_BundleId_exact = df_app_join_target.loc[mask]

df_app_search_BundleId_exact.head()
df_app_search_BundleId_exact.shape


'''
Merging app
'''

df_app_search_merge = pd.concat([df_app_search_Appname_exact, df_app_search_BundleId_exact])
dupe_col = ['PanelistId', 'Unix_StartTimeEvent_offset', 'AppName', 'BundleId', 'VisitDuration', 'activity']
df_app_search_merge = df_app_search_merge.drop_duplicates(subset=dupe_col)

df_app_search_merge.head()
df_app_search_merge.shape

## Activity -- Web Domain Search
**Web Domain Exact Match**

In [ ]:
'''
web domain shop
'''

# isolate list
weblistshop=[
    'Amazon',
    'GearBest',
    'TigerDirect',
    'MicroCenter',
    'BestBuy',
    'newegg',
    'eBay',
    'Alibaba',
    'Costco',
    'Walmart',
    'Apple',
    'Frys',
    'Office Depot',
    'Office Max',
    'Microsoft Store',
    'Staples',
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint ',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
    'iPhone',
    'Galaxy',
    'Note',
    'Pixel',
    'P30',
]

# lower case
weblistshop =  [x.lower() for x in weblistshop]

# apply filter
mask = df_web_join_target.PageDomain.str.contains('|'.join(weblistshop), na=False)
df_web_shop_search_name_exact = df_web_join_target.loc[mask]

df_web_shop_search_name_exact.head()
df_web_shop_search_name_exact.shape

## Activity - Web Search Term
**Exact Match**

In [ ]:
'''
Web Search Terms
'''

#create dictionary
webtermlist=[
    'Apple',
    'Samsung',
    'Huawei',
    'Google ',
    'LG',
    'Lenovo',
    'HMD',
    'Xiaomi',
    'Oppo',
    'Vivo',
    'Tecno',
    'Android',
    'iOS',
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint ',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
    'iPhone',
    'Galaxy',
    'Note',
    'Pixel',
    'P30',
    'Game of Thrones',
    'GOT',
    'Fantastic Beasts',
    'Netflix',
    'Promotions',
    'Discounts',
]


# lower case
webtermlist =  [x.lower() for x in webtermlist]

# apply filter
mask = df_web_join_target.SearchTerm.str.contains('|'.join(webtermlist), na=False)
df_web_search_term_exact = df_web_join_target.loc[mask]


df_web_search_term_exact.head()
df_web_search_term_exact.shape


## Activity -- Web Review

In [ ]:
# WEB review

'''
WEB Review search
'''
# isolate list
weblistrev=[
    'Gizmodo',
    'Wired',
    'ArsTechnica',
    'cnet',
    'consumerreports',
    'PC World',
    'PCMag',
    'The Verge',
    'TechAdvisor',
    'pricegrabber',
    'Google Shopping',
    'Nextag',
    'Shopping.com',
    'Shopzilla',
    'Bing Shopping',
    'Google.com\/shopping',
    'amazon',
    'amazon.c',
    'amazon.co',
    'amazon.com',
    'amazon.cp',
]


# lower case
weblistrev =  [x.lower() for x in weblistrev]

# apply app list
mask = df_web_join_target.PageDomain.str.contains('|'.join(weblistrev), na=False)
df_web_rev_search_name = df_web_join_target.loc[mask]

df_web_rev_search_name.head()
df_web_rev_search_name.shape


In [ ]:
# # import fuzzywuzzy library
# from fuzzywuzzy import fuzz

# '''
# Fuzzy Match function
# '''
# def fuzzymatch(dataframe, search_terms, source_column):
    
#     # fuzzy match
#     search_terms =  [x.lower() for x in applist]
#     search_terms = '|'.join(search_terms)
    
    
#     # fuzzy match application
#     dataframe[str(source_column)].fillna("", inplace=True)
    
#     token_ratio_score = []
    
#     for i, r in dataframe.iterrows():
#         str1= r[str(source_column)]
#         str2 = search_terms
#         token_ratio = fuzz.token_set_ratio(str1.lower(),str2)
#         token_ratio_score.append(token_ratio)
    
#     dataframe['token_ratio_score'] = token_ratio_score
    
#     return dataframe

# #search web search terms from list
# df_web_rev_search_name_fuzzy = fuzzymatch(
#     dataframe = df_web_join_survey_panel, 
#     search_terms = weblistrev, 
#     source_column = 'PageDomain'
# )



# # filtering dataframe
# mask = (df_web_rev_search_name_fuzzy['token_ratio_score']>=80)
# df_web_rev_search_name_fuzzy_cleaned = df_web_rev_search_name_fuzzy[mask]



# # preview dataframe
# df_web_rev_search_name_fuzzy_cleaned.head()
# df_web_rev_search_name_fuzzy_cleaned.shape

# df_web_rev_search_name_fuzzy_cleaned.loc[:,['PageDomain']].drop_duplicates(subset=['PageDomain']).to_csv('../WebReviewSearch_Fuzzy.csv')

## Activity -- Shopper term

In [ ]:
'''
SHOPPER
'''

#create dictionary
shoppertermlist=[
    'Apple',
    'Samsung',
    'Huawei',
    'Google ',
    'LG',
    'Lenovo',
    'HMD',
    'Xiaomi',
    'Oppo',
    'Vivo',
    'Tecno',
    'Android',
    'iOS',
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint ',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
    'iPhone',
    'Galaxy',
    'Note',
    'Pixel',
    'P30',
    'Game of Thrones',
    'GOT',
    'Fantastic Beasts',
    'Netflix',
    'Promotions',
    'Discounts',
]


# lower case
shoppertermlist = [x.lower() for x in shoppertermlist]


# apply filter
mask = df_shopper_join_target.SearchTerm.str.contains('|'.join(shoppertermlist), na=False)
df_shopper_search_term_exact = df_shopper_join_target.loc[mask]

df_shopper_search_term_exact.head()
df_shopper_search_term_exact.shape


## Activity -- Shopper product

In [ ]:
'''
SHOPPER AMAZON PRODUCTS
'''

#create dictionary
shopperprodlist=[
    'Apple',
    'Samsung',
    'Huawei',
    'Google',
    'LG',
    'Lenovo',
    'HMD',
    'Xiaomi',
    'Oppo',
    'Vivo',
    'Tecno',
    'Android',
    'iOS',
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
    'iPhone',
    'Galaxy',
    'Note',
    'Pixel',
    'P30',
]


# lower case
shopperprodlist = [x.lower() for x in shopperprodlist]


# apply filter
mask = df_shopper_join_target.ProductName.str.contains('|'.join(shopperprodlist), na=False)
df_shopper_search_prod_exact = df_shopper_join_target.loc[mask]


df_shopper_search_prod_exact.head()
df_shopper_search_prod_exact.shape

## ACTIVITY - VISITS 

In [ ]:
'''
PHYSICAL STORE VISITS
'''

visitstermlist=[
    'Apple Store',
    'Best Buy',
    'GameStop',
    'Microsoft Store',
    'Office Depot',
    'officemax',
    'Staples',
    'Frys Electronics',
    'P.C. Richard & Son',
    'apple baybrook',
    'staples warehouse',
    'apple sherman oaks',
    'apple wellington green',
    'staples® print & marketing services',
    "/^fry's electronics$/",
    "/^frys$/",
    'AT&T',
    'Verizon',
    'T-Mobile',
    'Cricket',
    'Boost',
    'Boost Mobile',
    'Metro',
    'Metro PCS',
    'Sprint',
    'US Cellular',
    'U.S. Cellular',
    'Vodafone',
    'Virgin',
    'Xfinity',
    'Target',
    'Walmart',
]


#str.lower(visitstermlistor)
visitstermlist =  [x.lower() for x in visitstermlist]


# apply filter
mask = visits_join.LocationName_Event.str.contains('|'.join(visitstermlist), na=False)
visits_search_term = visits_join.loc[mask]

visits_search_term.head()
visits_search_term.shape

## Aggregating and Appending final variables

**TO ADD**
* Search - rollup Web and Amazon search for each period 
* Delete - Amazon Visit column

**TO APPEND**
* Check Demos
* Extra P2P data from tabber

In [ ]:
'''
Copying -- Processed Dataframes
'''

# copies
visits_search_term_coded = visits_search_term.copy()
df_shopper_search_prod_coded = df_shopper_search_prod_exact.copy()
df_shopper_search_term_coded = df_shopper_search_term_exact.copy()
df_web_search_term_coded = df_web_search_term_exact.copy()
df_web_shop_search_name_coded = df_web_shop_search_name_exact.copy()
df_web_rev_search_name_coded = df_web_rev_search_name.copy()
df_app_search_name_coded = df_app_search_merge.copy()



# isolate columns
visits_search_term_coded = visits_search_term_coded.loc[:, ['PanelistId', 'activity']]
df_shopper_search_prod_coded = df_shopper_search_prod_coded.loc[:, ['PanelistId', 'activity']]
df_shopper_search_term_coded = df_shopper_search_term_coded.loc[:, ['PanelistId', 'activity']]
df_web_search_term_coded = df_web_search_term_coded.loc[:, ['PanelistId', 'activity']]
df_web_shop_search_name_coded = df_web_shop_search_name_coded.loc[:, ['PanelistId', 'activity']]
df_web_rev_search_name_coded = df_web_rev_search_name_coded.loc[:, ['PanelistId', 'activity']]
df_app_search_name_coded = df_app_search_name_coded.loc[:, ['PanelistId', 'activity']]



# coding event/activity type
visits_search_term_coded.insert(len(visits_search_term_coded.columns), column='Brick & Mortar Visit', value=1)
df_shopper_search_prod_coded.insert(len(df_shopper_search_prod_coded.columns), column='Amazon Visit', value=1)
df_shopper_search_term_coded.insert(len(df_shopper_search_term_coded.columns), column='Amazon Search', value=1)
df_web_search_term_coded.insert(len(df_web_search_term_coded.columns), column='Web Search', value=1)
df_web_shop_search_name_coded.insert(len(df_web_shop_search_name_coded.columns), column='Shopping Site Visit', value=1)
df_web_rev_search_name_coded.insert(len(df_web_rev_search_name_coded.columns), column='Review Site Visit', value=1)
df_app_search_name_coded.insert(len(df_app_search_name_coded.columns), column='App Visit', value=1)



'''
Visit Timeline
    - Brick and Morter visits
'''
# pre
pre_vis = visits_search_term_coded.query("activity == 'Pre'").copy()
pre_vis = pre_vis.drop_duplicates(subset=['PanelistId'])
pre_vis.pop('activity')
pre_vis['Pre Brick & Mortar Visit'] = pre_vis['Brick & Mortar Visit']
pre_vis.pop('Brick & Mortar Visit')

# during visits
dur_vis = visits_search_term_coded.query("activity == 'During'").copy()
dur_vis = dur_vis.drop_duplicates(subset=['PanelistId'])
dur_vis.pop('activity')
dur_vis['In-Brick & Mortar Visit'] = dur_vis['Brick & Mortar Visit']
dur_vis.pop('Brick & Mortar Visit')


# post visits
post_vis = visits_search_term_coded.query("activity == 'Post'").copy()
post_vis = post_vis.drop_duplicates(subset=['PanelistId'])
post_vis.pop('activity')
post_vis['PostBrick & Mortar Visit'] = post_vis['Brick & Mortar Visit']
post_vis.pop('Brick & Mortar Visit')




'''
Visit Timeline
    - Amazon Visit
'''
# pre
pre_shopvis = df_shopper_search_prod_coded.query("activity == 'Pre'").copy()
pre_shopvis = pre_shopvis.drop_duplicates(subset=['PanelistId'])
pre_shopvis.pop('activity')
pre_shopvis['Pre Amazon Visit'] = pre_shopvis['Amazon Visit']
pre_shopvis.pop('Amazon Visit')

# during visits
dur_shopvis = df_shopper_search_prod_coded.query("activity == 'During'").copy()
dur_shopvis = dur_shopvis.drop_duplicates(subset=['PanelistId'])
dur_shopvis.pop('activity')
dur_shopvis['In-Amazon Visit'] = dur_shopvis['Amazon Visit']
dur_shopvis.pop('Amazon Visit')


# post visits
post_shopvis = df_shopper_search_prod_coded.query("activity == 'Post'").copy()
post_shopvis = post_shopvis.drop_duplicates(subset=['PanelistId'])
post_shopvis.pop('activity')
post_shopvis['PostAmazon Visit'] = post_shopvis['Amazon Visit']
post_shopvis.pop('Amazon Visit')




'''
Visit Timeline
    - Amazon Search
'''
# pre
pre_shopsearch = df_shopper_search_term_coded.query("activity == 'Pre'").copy()
pre_shopsearch = pre_shopsearch.drop_duplicates(subset=['PanelistId'])
pre_shopsearch.pop('activity')
pre_shopsearch['Pre Amazon Search'] = pre_shopsearch['Amazon Search']
pre_shopsearch.pop('Amazon Search')

# during visits
dur_shopsearch = df_shopper_search_term_coded.query("activity == 'During'").copy()
dur_shopsearch = dur_shopsearch.drop_duplicates(subset=['PanelistId'])
dur_shopsearch.pop('activity')
dur_shopsearch['In-Amazon Search'] = dur_shopsearch['Amazon Search']
dur_shopsearch.pop('Amazon Search')


# post visits
post_shopsearch = df_shopper_search_term_coded.query("activity == 'Post'").copy()
post_shopsearch = post_shopsearch.drop_duplicates(subset=['PanelistId'])
post_shopsearch.pop('activity')
post_shopsearch['PostAmazon Search'] = post_shopsearch['Amazon Search']
post_shopsearch.pop('Amazon Search')




'''
Visit Timeline
    - Web Search
'''
# pre
pre_websearch = df_web_search_term_coded.query("activity == 'Pre'").copy()
pre_websearch = pre_websearch.drop_duplicates(subset=['PanelistId'])
pre_websearch.pop('activity')
pre_websearch['Pre Web Search'] = pre_websearch['Web Search']
pre_websearch.pop('Web Search')

# during visits
dur_websearch = df_web_search_term_coded.query("activity == 'During'").copy()
dur_websearch = dur_websearch.drop_duplicates(subset=['PanelistId'])
dur_websearch.pop('activity')
dur_websearch['In-Web Search'] = dur_websearch['Web Search']
dur_websearch.pop('Web Search')


# post visits
post_websearch = df_web_search_term_coded.query("activity == 'Post'").copy()
post_websearch = post_websearch.drop_duplicates(subset=['PanelistId'])
post_websearch.pop('activity')
post_websearch['PostWeb Search'] = post_websearch['Web Search']
post_websearch.pop('Web Search')



'''
Visit Timeline
    - Shopping Site Visit
'''
# pre
pre_shopsitevis = df_web_shop_search_name_coded.query("activity == 'Pre'").copy()
pre_shopsitevis = pre_shopsitevis.drop_duplicates(subset=['PanelistId'])
pre_shopsitevis.pop('activity')
pre_shopsitevis['Pre Shopping Site Visit'] = pre_shopsitevis['Shopping Site Visit']
pre_shopsitevis.pop('Shopping Site Visit')

# during visits
dur_shopsitevis = df_web_shop_search_name_coded.query("activity == 'During'").copy()
dur_shopsitevis = dur_shopsitevis.drop_duplicates(subset=['PanelistId'])
dur_shopsitevis.pop('activity')
dur_shopsitevis['In-Shopping Site Visit'] = dur_shopsitevis['Shopping Site Visit']
dur_shopsitevis.pop('Shopping Site Visit')


# post visits
post_shopsitevis = df_web_shop_search_name_coded.query("activity == 'Post'").copy()
post_shopsitevis = post_shopsitevis.drop_duplicates(subset=['PanelistId'])
post_shopsitevis.pop('activity')
post_shopsitevis['PostShopping Site Visit'] = post_shopsitevis['Shopping Site Visit']
post_shopsitevis.pop('Shopping Site Visit')



'''
Visit Timeline
    - Review Site Visit
'''
# pre
pre_revsitevis = df_web_rev_search_name_coded.query("activity == 'Pre'").copy()
pre_revsitevis = pre_revsitevis.drop_duplicates(subset=['PanelistId'])
pre_revsitevis.pop('activity')
pre_revsitevis['Pre Review Site Visit'] = pre_revsitevis['Review Site Visit']
pre_revsitevis.pop('Review Site Visit')

# during visits
dur_revsitevis = df_web_rev_search_name_coded.query("activity == 'During'").copy()
dur_revsitevis = dur_revsitevis.drop_duplicates(subset=['PanelistId'])
dur_revsitevis.pop('activity')
dur_revsitevis['In-Review Site Visit'] = dur_revsitevis['Review Site Visit']
dur_revsitevis.pop('Review Site Visit')


# post visits
post_revsitevis = df_web_rev_search_name_coded.query("activity == 'Post'").copy()
post_revsitevis = post_revsitevis.drop_duplicates(subset=['PanelistId'])
post_revsitevis.pop('activity')
post_revsitevis['PostReview Site Visit'] = post_revsitevis['Review Site Visit']
post_revsitevis.pop('Review Site Visit')



'''
Visit Timeline
    - Visit Search
'''
# pre
pre_appsearch = df_app_search_name_coded.query("activity == 'Pre'").copy()
pre_appsearch = pre_appsearch.drop_duplicates(subset=['PanelistId'])
pre_appsearch.pop('activity')
pre_appsearch['Pre App Visit'] = pre_appsearch['App Visit']
pre_appsearch.pop('App Visit')

# during visits
dur_appsearch = df_app_search_name_coded.query("activity == 'During'").copy()
dur_appsearch = dur_appsearch.drop_duplicates(subset=['PanelistId'])
dur_appsearch.pop('activity')
dur_appsearch['In-App Visit'] = dur_appsearch['App Visit']
dur_appsearch.pop('App Visit')


# post visits
post_appsearch = df_app_search_name_coded.query("activity == 'Post'").copy()
post_appsearch = post_appsearch.drop_duplicates(subset=['PanelistId'])
post_appsearch.pop('activity')
post_appsearch['PostApp Visit'] = post_appsearch['App Visit']
post_appsearch.pop('App Visit')



In [ ]:
# stacking dataframes
dataframes = [visits_search_term_coded, df_shopper_search_prod_coded, df_shopper_search_term_coded, df_web_search_term_coded, df_web_shop_search_name_coded, df_web_rev_search_name_coded, df_app_search_name_coded]    # dataframes stacked
pid_og = pd.concat(dataframes)    # stacking dataframes


# isolate PIDs
pid_og = pid_og.loc[:, ['PanelistId']]


# adding missing PIDs
visits_survey_join_pid = df_target_visits.loc[:, ['PanelistId']].drop_duplicates(subset=['PanelistId'])    # import full canonical PID list
pid = pd.concat([pid_og, visits_survey_join_pid])    # stacking both lists of PIDs
pid = pid.drop_duplicates(subset=['PanelistId'])    # dropping duplicates


# importing demographic variables
#dems = pd.read_csv('../Demographics/MsoftDig_PID_Demos.csv')    # import dem csv
#dems = dems.rename(columns={'pid':'PanelistId'})    # rename pid to PanelistId for merge


# merging demographics and 
#pid = pid.merge(dems, how='left', on='PanelistId')    # merging PID and dems dataframes

'''
Merging pre visit
'''

# merging PIDs to relevant variables
pid = pid.merge(pre_vis, how='left', on='PanelistId')\
.merge(pre_shopvis, how='left', on='PanelistId')\
.merge(pre_shopsearch, how='left', on='PanelistId')\
.merge(pre_websearch, how='left', on='PanelistId')\
.merge(pre_shopsitevis, how='left', on='PanelistId')\
.merge(pre_revsitevis, how='left', on='PanelistId')\
.merge(pre_appsearch, how='left', on='PanelistId')


# recoding search - 2w Web search | Amazon search 
mask = ((pid['Pre Amazon Search'] == 1)|(pid['Pre Web Search'] == 1))
pid.loc[mask, 'Pre Search'] = 1

# recoding search - 2w Web search | Amazon search 
mask = ((pid['Pre Shopping Site Visit'] == 1)|(pid['Pre Review Site Visit'] == 1)|(pid['Pre App Visit'] == 1)|(pid['Pre Amazon Search'] == 1)|(pid['Pre Web Search'] == 1))
pid.loc[mask, 'Pre Digital Activity'] = 1



'''
Merging during visit
'''

pid = pid.merge(dur_vis, how='left', on='PanelistId')\
.merge(dur_shopvis, how='left', on='PanelistId')\
.merge(dur_shopsearch, how='left', on='PanelistId')\
.merge(dur_websearch, how='left', on='PanelistId')\
.merge(dur_shopsitevis, how='left', on='PanelistId')\
.merge(dur_revsitevis, how='left', on='PanelistId')\
.merge(dur_appsearch, how='left', on='PanelistId')


# recoding search - in Web search | Amazon search 
mask = ((pid['In-Amazon Search'] == 1)|(pid['In-Web Search'] == 1))
pid.loc[mask, 'In-Search'] = 1


# recoding search - in Web search | Amazon search 
mask = ((pid['In-Shopping Site Visit'] == 1)|(pid['In-Review Site Visit'] == 1)|(pid['In-App Visit'] == 1)|(pid['In-Amazon Search'] == 1)|(pid['In-Web Search'] == 1))
pid.loc[mask, 'In-Digital Activity'] = 1



'''
Merging post visit
'''

pid = pid.merge(post_vis, how='left', on='PanelistId')\
.merge(post_shopvis, how='left', on='PanelistId')\
.merge(post_shopsearch, how='left', on='PanelistId')\
.merge(post_websearch, how='left', on='PanelistId')\
.merge(post_shopsitevis, how='left', on='PanelistId')\
.merge(post_revsitevis, how='left', on='PanelistId')\
.merge(post_appsearch, how='left', on='PanelistId')

# recoding search - post Web search | Amazon search 
mask = ((pid['PostAmazon Search'] == 1)|(pid['PostWeb Search'] == 1))
pid.loc[mask, 'PostSearch'] = 1


# recoding search - post Web search | Amazon search 
mask = ((pid['PostShopping Site Visit'] == 1)|(pid['PostReview Site Visit'] == 1)|(pid['PostApp Visit'] == 1)|(pid['PostAmazon Search'] == 1)|(pid['PostWeb Search'] == 1))
pid.loc[mask, 'PostDigital Activity'] = 1


# replace null values
pid = pid.fillna('')
pid = pid.replace('', 0)

pid.pop('Pre Amazon Visit')
pid.pop('In-Amazon Visit')
pid.pop('PostAmazon Visit')


'''
APPENDING SURVEY DATA
'''
'''
# recoded
val = {'Male':1, 'Female':2}
pid['gender'] = pid['gender'].replace(to_replace=val)

val = {'Less than $25,000':1, '$25,000 to $34,999':2, '$35,000 to $49,999':3, '$50,000 to $74,999':4, '$75,000 to $99,999':5, '$100,000 or more':6}
pid['income'] = pid['income'].replace(to_replace=val)



# survey data
survey = pd.read_csv('../SurveyData/MFour - Microsoft - MicrosoftDigitalDataPilot - Survey 1 Digital Telemetry Data.V2 with new variables 03 28 19.csv')
pid_survey = pid.merge(survey, how='left', right_on=['Panelist_Id'], left_on=['PanelistId'])
pid_survey.pop('Panelist_Id')

# 
nets = pd.read_csv('../SurveyData/Nets_Topline/Nets_MFour - Microsoft - MicrosoftDigitalDataPilot - Survey 1 Digital Telemetry Data.V2 with new MFour variables 03 29 19(b).csv')
pid_survey = pid_survey.merge(nets, how='left', right_on=['Panelist_Id'], left_on=['PanelistId'])
'''

'''
EXPORTING MERGED FILE
'''
# EXPORTING MERGED DATA
pid.to_csv('032919-AT&T_DigVis.csv', index=False)

In [ ]:
pid_survey.income

In [ ]:
'''
PLACEHOLDER - INSERT LOCATION VISIT CODING WHEN AVAILABLE 
'''

## REPORTING FUNCTIONS

In [ ]:
pid = pid_survey.copy()

## ALL VISITS

**Brick and Mortar Store Visit Numbers**
* Past 2 weeks - 364/444 ~= 82% (0.8198198198)
* Post - 42/444 ~=  9% (0.09459459459)

In [ ]:
pid.groupby(['Pre Brick & Mortar Visit'])['PanelistId'].count()/276*100

pid.groupby(['In-Brick & Mortar Visit'])['PanelistId'].count()/276*100

pid.groupby(['PostBrick & Mortar Visit'])['PanelistId'].count()/276*100

In [ ]:
pid.groupby(['Pre Brick & Mortar Visit'])['PanelistId'].count()

pid.groupby(['In-Brick & Mortar Visit'])['PanelistId'].count()

pid.groupby(['PostBrick & Mortar Visit'])['PanelistId'].count()

**Amazon Visit**

In [ ]:
# pid.groupby(['2w Amazon Visit'])['PanelistId'].count()/444*100

# pid.groupby(['In-Amazon Visit'])['PanelistId'].count()/444*100

# pid.groupby(['PostAmazon Visit'])['PanelistId'].count()/444*100

**Amazon Search**

In [ ]:
pid.groupby(['Pre Amazon Search'])['PanelistId'].count()

pid.groupby(['In-Amazon Search'])['PanelistId'].count()

pid.groupby(['PostAmazon Search'])['PanelistId'].count()

In [ ]:
pid.groupby(['Pre Amazon Search'])['PanelistId'].count()/276*100

pid.groupby(['In-Amazon Search'])['PanelistId'].count()/276*100

pid.groupby(['PostAmazon Search'])['PanelistId'].count()/276*100

**Web Search**

In [ ]:
pid.groupby(['Pre Web Search'])['PanelistId'].count()/276*100

pid.groupby(['In-Web Search'])['PanelistId'].count()/276*100

pid.groupby(['PostWeb Search'])['PanelistId'].count()/276*100

**Shopping Site Visit**

In [ ]:
pid.groupby(['Pre Shopping Site Visit'])['PanelistId'].count()/276*100

pid.groupby(['In-Shopping Site Visit'])['PanelistId'].count()/276*100

pid.groupby(['PostShopping Site Visit'])['PanelistId'].count()/276*100

**Review Site Visit**

In [ ]:
pid.groupby(['Pre Review Site Visit'])['PanelistId'].count()/276*100

pid.groupby(['In-Review Site Visit'])['PanelistId'].count()/276*100

pid.groupby(['PostReview Site Visit'])['PanelistId'].count()/276*100

**App Visit**

In [ ]:
pid.groupby(['Pre App Visit'])['PanelistId'].count()/276*100

pid.groupby(['In-App Visit'])['PanelistId'].count()/276*100

pid.groupby(['PostApp Visit'])['PanelistId'].count()/276*100

## TOP 5 LOCATIONS

In [ ]:
visits_search_term.activity.value_counts()

In [ ]:
visits_search_term.groupby(['activity']).LocationName_Event.value_counts()




In [ ]:
visits_search_term.groupby(['activity']).LocationName_Event.value_counts(normalize=True)*100

In [ ]:
df_web_shop_search_name_exact.PanelistId.nunique()

In [ ]:
df_web_shop_search_name_exact

In [ ]:
digital_post.groupby(['PostShopping Site Visit'])['PanelistId'].count()

In [ ]:
pre_shopping_site = digital_pre.loc[digital_pre['2w Shopping Site Visit'] == 1]
pre_shopping_site = df_web_shop_search_name_exact.merge(pre_shopping_site, how='right', on='PanelistId')

regex = 'best|costco|amazon|apple|micro|walmart|jet'
mask = pre_shopping_site.PageDomain.str.contains(regex, case=False, na=False)
pre_web_domain = pre_shopping_site.loc[mask]


pre_web_domain.groupby(['activity', 'PageDomain',]).PanelistId.nunique()

In [ ]:
pre_web_domain.groupby(['activity', 'PageDomain',]).PanelistId.nunique().to_csv('../4Amazon_Pre_Pagedomain.csv')

In [ ]:

post_shopping_site = digital_post.loc[digital_post['PostShopping Site Visit'] == 1]
post_shopping_site = df_web_shop_search_name_exact.merge(digital_post, how='right', on='PanelistId')


regex = 'best|costco|amazon|apple|micro|walmart|jet'
mask = post_shopping_site.PageDomain.str.contains(regex, case=False, na=False)
post_web_domain = post_shopping_site.loc[mask]

post_web_domain.groupby(['activity', 'PageDomain']).PanelistId.nunique()

In [ ]:
post_web_domain.groupby(['activity', 'PageDomain']).PanelistId.nunique().to_csv('../4Amazon_Post_Pagedomain.csv')

In [ ]:
regex = 'best|costco|amazon|apple|micro'
mask = df_web_shop_search_name_exact.PageDomain.str.contains(regex, case=False, na=False)
web_domain = df_web_shop_search_name_exact.loc[mask]


web_domain.groupby(['activity']).PageDomain.value_counts()




web_domain.groupby(['activity']).PanelistId.nunique()


In [ ]:
web_domain.groupby(['activity']).PageDomain.value_counts(normalize=True)*100

In [ ]:
df_web_shop_search_name_exact.groupby(['activity']).PageDomain.value_counts(normalize=True)*100

## REPORTING - DIGITAL CUTS OF DATA

## Digital - All

In [ ]:
# all
mask = (
    (pid['Pre Amazon Search'] == 1)|\
    (pid['Pre Amazon Visit'] == 1)|\
    (pid['Pre Web Search'] == 1)|\
    (pid['Pre Shopping Site Visit'] == 1)|\
    (pid['Pre Review Site Visit'] == 1)|\
    (pid['Pre App Visit'] == 1)|\
    (pid['In-Amazon Search'] == 1)|\
    (pid['In-Amazon Visit'] == 1)|\
    (pid['In-Web Search'] == 1)|\
    (pid['In-Shopping Site Visit'] == 1)|\
    (pid['In-Review Site Visit'] == 1)|\
    (pid['In-App Visit'] == 1)|\
    (pid['PostAmazon Search'] == 1)|\
    (pid['PostAmazon Visit'] == 1)|\
    (pid['PostWeb Search'] == 1)|\
    (pid['PostShopping Site Visit'] == 1)|\
    (pid['PostReview Site Visit'] == 1)|\
    (pid['PostApp Visit'] == 1)
)


digital_all = pid.loc[mask]

digital_all.shape

## Digital - pre

In [ ]:


# pre
mask = (
    (pid['Pre Amazon Search'] == 1)|\
    (pid['Pre Web Search'] == 1)|\
    (pid['Pre Shopping Site Visit'] == 1)|\
    (pid['Pre Review Site Visit'] == 1)|\
    (pid['Pre App Visit'] == 1)
)

digital_pre = pid.loc[mask]


digital_pre.shape

**Brick and Mortar Store Visit Numbers**
* Past 2 weeks - 364/444 ~= 82% (0.8198198198)
* Post - 42/444 ~=  9% (0.09459459459)

In [ ]:
digital_pre.groupby(['2w Brick & Mortar Visit'])['PanelistId'].count()/231*100

digital_pre.groupby(['In-Brick & Mortar Visit'])['PanelistId'].count()/231*100

digital_pre.groupby(['PostBrick & Mortar Visit'])['PanelistId'].count()/231*100

**Amazon Visit**

In [ ]:
# digital_pre.groupby(['2w Amazon Visit'])['PanelistId'].count()/231*100

# digital_pre.groupby(['In-Amazon Visit'])['PanelistId'].count()/231*100

# digital_pre.groupby(['PostAmazon Visit'])['PanelistId'].count()/231*100

**Amazon Search**

In [ ]:
digital_pre.groupby(['2w Amazon Search'])['PanelistId'].count()/231*100

digital_pre.groupby(['In-Amazon Search'])['PanelistId'].count()/231*100

digital_pre.groupby(['PostAmazon Search'])['PanelistId'].count()/231*100

**Web Search**

In [ ]:
digital_pre.groupby(['2w Web Search'])['PanelistId'].count()/231*100

digital_pre.groupby(['In-Web Search'])['PanelistId'].count()/231*100

digital_pre.groupby(['PostWeb Search'])['PanelistId'].count()/231*100

**Shopping Site Visit**

In [ ]:
digital_pre.groupby(['2w Shopping Site Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['In-Shopping Site Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['PostShopping Site Visit'])['PanelistId'].count()/277*100

In [ ]:
digital_pre.groupby(['2w Shopping Site Visit'])['PanelistId'].count()

digital_pre.groupby(['In-Shopping Site Visit'])['PanelistId'].count()

digital_pre.groupby(['PostShopping Site Visit'])['PanelistId'].count()

**Review Site Visit**

In [ ]:
digital_pre.groupby(['2w Review Site Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['In-Review Site Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['PostReview Site Visit'])['PanelistId'].count()/277*100

In [ ]:
digital_pre.groupby(['2w Review Site Visit'])['PanelistId'].count()

digital_pre.groupby(['In-Review Site Visit'])['PanelistId'].count()

digital_pre.groupby(['PostReview Site Visit'])['PanelistId'].count()

**App Visit**

In [ ]:
digital_pre.groupby(['2w App Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['In-App Visit'])['PanelistId'].count()/277*100

digital_pre.groupby(['PostApp Visit'])['PanelistId'].count()/277*100

In [ ]:
digital_pre.groupby(['2w App Visit'])['PanelistId'].count()

digital_pre.groupby(['In-App Visit'])['PanelistId'].count()

digital_pre.groupby(['PostApp Visit'])['PanelistId'].count()

In [ ]:
digital_pre.groupby(['2w Search'])['PanelistId'].count()

In [ ]:
digital_pre.groupby(['2w Search'])['PanelistId'].count()

In [ ]:
list(digital_pre.columns)

## Digital - during

In [ ]:
# during
mask = (
    (pid['In-Amazon Search'] == 1)|\
    (pid['In-Web Search'] == 1)|\
    (pid['In-Shopping Site Visit'] == 1)|\
    (pid['In-Review Site Visit'] == 1)|\
    (pid['In-App Visit'] == 1)
)

digital_dur = pid.loc[mask]


digital_dur.shape

In [ ]:
digital_dur.groupby(['In-Search'])['PanelistId'].count()

In [ ]:
list(digital_dur.columns)

**Brick and Mortar Store Visit Numbers**
* Past 2 weeks - 364/444 ~= 82% (0.8198198198)
* Post - 42/444 ~=  9% (0.09459459459)

In [ ]:
digital_dur.groupby(['2w Brick & Mortar Visit'])['PanelistId'].count()/9*100

digital_dur.groupby(['In-Brick & Mortar Visit'])['PanelistId'].count()/9*100

digital_dur.groupby(['PostBrick & Mortar Visit'])['PanelistId'].count()/9*100

**Amazon Visit**

In [ ]:
# digital_dur.groupby(['2w Amazon Visit'])['PanelistId'].count()/9*100

# digital_dur.groupby(['In-Amazon Visit'])['PanelistId'].count()/9*100

# digital_dur.groupby(['PostAmazon Visit'])['PanelistId'].count()/9*100

**Amazon Search**

In [ ]:
digital_dur.groupby(['2w Amazon Search'])['PanelistId'].count()/9*100

digital_dur.groupby(['In-Amazon Search'])['PanelistId'].count()/9*100

digital_dur.groupby(['PostAmazon Search'])['PanelistId'].count()/9*100

**Web Search**

In [ ]:
digital_dur.groupby(['2w Web Search'])['PanelistId'].count()/9*100

digital_dur.groupby(['In-Web Search'])['PanelistId'].count()/9*100

digital_dur.groupby(['PostWeb Search'])['PanelistId'].count()/9*100

**Shopping Site Visit**

In [ ]:
digital_dur.groupby(['2w Shopping Site Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['In-Shopping Site Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['PostShopping Site Visit'])['PanelistId'].count()/37*100

**Review Site Visit**

In [ ]:
digital_dur.groupby(['2w Review Site Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['In-Review Site Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['PostReview Site Visit'])['PanelistId'].count()/37*100

**App Visit**

In [ ]:
digital_dur.groupby(['2w App Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['In-App Visit'])['PanelistId'].count()/37*100

digital_dur.groupby(['PostApp Visit'])['PanelistId'].count()/37*100

## Digital - post

In [ ]:
# post
mask = (
    (pid['PostAmazon Search'] == 1)|\
    (pid['PostWeb Search'] == 1)|\
    (pid['PostShopping Site Visit'] == 1)|\
    (pid['PostReview Site Visit'] == 1)|\
    (pid['PostApp Visit'] == 1)
)

digital_post = pid.loc[mask]

digital_post.shape

In [ ]:
digital_post.groupby(['PostSearch'])['PanelistId'].count()

In [ ]:
list(digital_post.columns)

In [ ]:
digital_post.groupby(['2w Brick & Mortar Visit'])['PanelistId'].count()/24*100

digital_post.groupby(['In-Brick & Mortar Visit'])['PanelistId'].count()/24*100

digital_post.groupby(['PostBrick & Mortar Visit'])['PanelistId'].count()/24*100

**Amazon Visit**

In [ ]:
# digital_post.groupby(['2w Amazon Visit'])['PanelistId'].count()/24*100

# digital_post.groupby(['In-Amazon Visit'])['PanelistId'].count()/24*100

# digital_post.groupby(['PostAmazon Visit'])['PanelistId'].count()/24*100

**Amazon Search**

In [ ]:
digital_post.groupby(['2w Amazon Search'])['PanelistId'].count()/24*100

digital_post.groupby(['In-Amazon Search'])['PanelistId'].count()/24*100

digital_post.groupby(['PostAmazon Search'])['PanelistId'].count()/24*100

**Web Search**

In [ ]:
digital_post.groupby(['2w Web Search'])['PanelistId'].count()/24*100

digital_post.groupby(['In-Web Search'])['PanelistId'].count()/24*100

digital_post.groupby(['PostWeb Search'])['PanelistId'].count()/24*100

**Shopping Site Visit**

In [ ]:
digital_post.groupby(['2w Shopping Site Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['In-Shopping Site Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['PostShopping Site Visit'])['PanelistId'].count()/51*100

**Review Site Visit**

In [ ]:
digital_post.groupby(['2w Review Site Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['In-Review Site Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['PostReview Site Visit'])['PanelistId'].count()/51*100

**App Visit**

In [ ]:
digital_post.groupby(['2w App Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['In-App Visit'])['PanelistId'].count()/51*100

digital_post.groupby(['PostApp Visit'])['PanelistId'].count()/51*100